# Plant Disease Detection: Custom CNN vs. Transfer Learning (MobileNetV2, ResNet50) with Grad-CAM
**Dataset:** PlantVillage (54,303 leaf images, 38 classes) | **Framework:** TensorFlow / Keras

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then set `QUICK_TEST` below and Runtime → Run all.

In [ ]:
# Plant Disease Detection using CNN, MobileNetV2 and ResNet50
# 1) Setup
!pip install -q gradio

import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

QUICK_TEST = False      # True = tiny run (~3 min) to check everything works; False = full run
IMG = 160               # image size (160x160)
BATCH = 32
EPOCHS_CNN = 2 if QUICK_TEST else 10
EPOCHS_TL = 1 if QUICK_TEST else 5
os.makedirs("results", exist_ok=True)
tf.random.set_seed(42)

## 2) Load the PlantVillage dataset (80% train / 10% val / 10% test)

In [ ]:
# 2) Data loading
# Download only the colour images from the official PlantVillage GitHub repo (~3-5 min)
if not os.path.exists("PlantVillage-Dataset"):
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset
    !cd PlantVillage-Dataset && git sparse-checkout set raw/color

DATA_DIR = "PlantVillage-Dataset/raw/color"
folders = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
class_names = [n.replace("___", " - ").replace("_", " ") for n in folders]
NUM_CLASSES = len(class_names)

paths, labels = [], []
for i, folder in enumerate(folders):
    for f in os.listdir(os.path.join(DATA_DIR, folder)):
        paths.append(os.path.join(DATA_DIR, folder, f))
        labels.append(i)

# Shuffle once, then split 80 / 10 / 10
rng = np.random.default_rng(42)
order = rng.permutation(len(paths))
paths, labels = np.array(paths)[order], np.array(labels)[order]
if QUICK_TEST:
    paths, labels = paths[:2100], labels[:2100]
n = len(paths)
a, b = int(0.8 * n), int(0.9 * n)

def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return img, label

def to_ds(p, l):
    return tf.data.Dataset.from_tensor_slices((p, l)).map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

train_raw, val_raw, test_raw = to_ds(paths[:a], labels[:a]), to_ds(paths[a:b], labels[a:b]), to_ds(paths[b:], labels[b:])

print("Classes:", NUM_CLASSES)
print("Images used:", n, "| train:", a, "| val:", b - a, "| test:", n - b)

# Show a few sample leaves
plt.figure(figsize=(12, 4))
for i, (img, lbl) in enumerate(train_raw.take(6)):
    plt.subplot(1, 6, i + 1)
    plt.imshow(img)
    plt.title(class_names[int(lbl)], fontsize=7)
    plt.axis("off")
plt.savefig("results/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 3) Data augmentation + pipeline (reduces overfitting)

In [ ]:
# 3) Data augmentation and input pipeline
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def resize(img, lbl):
    return tf.image.resize(img, (IMG, IMG)), lbl

# Each model expects its own input scaling
PREPROCESS = {
    "Custom CNN": lambda x: x / 255.0,
    "MobileNetV2": tf.keras.applications.mobilenet_v2.preprocess_input,
    "ResNet50": tf.keras.applications.resnet50.preprocess_input,
}

def make_ds(raw, prep, train=False):
    ds = raw.map(resize, num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.shuffle(1000)
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (prep(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

## 4) The three models

In [ ]:
# 4) Models
# Model A: Custom CNN from scratch (Conv -> BatchNorm -> MaxPool blocks, Dropout, Softmax)
def build_cnn():
    return models.Sequential([
        layers.Input((IMG, IMG, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu", name="last_conv"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ])

# Model B and C: Transfer learning - pretrained ImageNet base is frozen, only a new head is trained
def build_transfer(base_fn):
    base = base_fn(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return models.Model(base.input, out)

BUILDERS = {
    "Custom CNN": build_cnn,
    "MobileNetV2": lambda: build_transfer(tf.keras.applications.MobileNetV2),
    "ResNet50": lambda: build_transfer(tf.keras.applications.ResNet50),
}
LAST_CONV = {"Custom CNN": "last_conv", "MobileNetV2": "out_relu", "ResNet50": "conv5_block3_out"}

## 5) Train all three models (Adam, sparse categorical cross-entropy, early stopping)

In [ ]:
# 5) Training
results, histories, trained, test_sets = [], {}, {}, {}

for name in BUILDERS:
    print(f"\n========== Training {name} ==========")
    prep = PREPROCESS[name]
    train_ds = make_ds(train_raw, prep, train=True)
    val_ds = make_ds(val_raw, prep)
    test_ds = make_ds(test_raw, prep)

    model = BUILDERS[name]()
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1),
    ]
    epochs = EPOCHS_CNN if name == "Custom CNN" else EPOCHS_TL

    start = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks)
    train_time = (time.time() - start) / 60

    loss, acc = model.evaluate(test_ds, verbose=0)
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    results.append({"Model": name,
                    "Test Accuracy (%)": round(acc * 100, 2),
                    "Test Loss": round(loss, 4),
                    "Total Params (M)": round(model.count_params() / 1e6, 2),
                    "Trainable Params (M)": round(trainable / 1e6, 2),
                    "Epochs Run": len(hist.history["loss"]),
                    "Train Time (min)": round(train_time, 1)})
    histories[name], trained[name], test_sets[name] = hist.history, model, test_ds
    print(f"{name}: test accuracy = {acc*100:.2f}%")

## 6) Comparison of the three models

In [ ]:
# 6) Results comparison
df = pd.DataFrame(results)
df.to_csv("results/comparison.csv", index=False)
print(df.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, h in histories.items():
    ax[0].plot(h["val_accuracy"], marker="o", label=name)
    ax[1].plot(h["val_loss"], marker="o", label=name)
ax[0].set_title("Validation Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].set_title("Validation Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.savefig("results/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Train vs validation accuracy for each model (shows under/over-fitting)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, h) in enumerate(histories.items()):
    ax[i].plot(h["accuracy"], label="train")
    ax[i].plot(h["val_accuracy"], label="val")
    ax[i].set_title(name); ax[i].set_xlabel("Epoch"); ax[i].legend()
plt.savefig("results/train_vs_val.png", dpi=150, bbox_inches="tight")
plt.show()

## 7) Detailed evaluation of the best model (precision, recall, F1, confusion matrix)

In [ ]:
# 7) Best model evaluation
best_name = df.loc[df["Test Accuracy (%)"].idxmax(), "Model"]
best = trained[best_name]
print("Best model:", best_name)

y_true, y_pred = [], []
for x, y in test_sets[best_name]:
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(best.predict(x, verbose=0), axis=1))

labels = list(range(NUM_CLASSES))
report = classification_report(y_true, y_pred, labels=labels, target_names=class_names, digits=3, zero_division=0)
print(report)
with open("results/classification_report.txt", "w") as f:
    f.write(f"Best model: {best_name}\n\n" + report)

cm = confusion_matrix(y_true, y_pred, labels=labels)
plt.figure(figsize=(14, 12))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(labels, class_names, rotation=90, fontsize=6)
plt.yticks(labels, class_names, fontsize=6)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix - {best_name}")
plt.savefig("results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 8) Grad-CAM: where on the leaf does the model look?

In [ ]:
# 8) Grad-CAM explainability
def grad_cam(model, img, layer_name):
    grad_model = tf.keras.Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img[None, ...])
        cls = int(tf.argmax(preds[0]))
        score = preds[0, cls]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv_out[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (IMG, IMG))[..., 0]
    return cam.numpy(), cls

plt.figure(figsize=(16, 8))
for i, (img, lbl) in enumerate(test_raw.take(4)):
    img = tf.image.resize(img, (IMG, IMG))
    cam, cls = grad_cam(best, PREPROCESS[best_name](img), LAST_CONV[best_name])
    plt.subplot(2, 4, i + 1)
    plt.imshow(img.numpy().astype("uint8"))
    plt.title("True: " + class_names[int(lbl)], fontsize=8); plt.axis("off")
    plt.subplot(2, 4, i + 5)
    plt.imshow(img.numpy().astype("uint8"))
    plt.imshow(cam, cmap="jet", alpha=0.4)
    plt.title("Pred: " + class_names[cls], fontsize=8); plt.axis("off")
plt.suptitle(f"Grad-CAM ({best_name})")
plt.savefig("results/gradcam.png", dpi=150, bbox_inches="tight")
plt.show()

## 9) Save the best model + predict on any leaf image

In [ ]:
# 9) Save model and predict function
best.save("results/best_model.keras")

def predict(image):
    img = tf.image.resize(tf.convert_to_tensor(image, dtype=tf.float32), (IMG, IMG))
    probs = best.predict(PREPROCESS[best_name](img)[None, ...], verbose=0)[0]
    return {class_names[i]: float(probs[i]) for i in np.argsort(probs)[::-1][:3]}

sample_img, sample_lbl = next(iter(test_raw.take(1)))
print("True:", class_names[int(sample_lbl)])
print("Predicted:", predict(sample_img.numpy()))

## 10) Web demo (Gradio): upload a leaf photo and get the diagnosis

In [ ]:
# 10) Gradio demo
import gradio as gr

gr.Interface(fn=predict,
             inputs=gr.Image(type="numpy", label="Upload a leaf image"),
             outputs=gr.Label(num_top_classes=3, label="Prediction"),
             title="Plant Disease Detection",
             description=f"Model: {best_name} trained on PlantVillage (38 classes)").launch(share=True)

## 11) Download all results (plots, table, report, model)

In [ ]:
# 11) Save 12 unseen test leaves for the live demo, then download everything as a zip
import shutil
os.makedirs("results/demo_images", exist_ok=True)
for i, (pth, lbl) in enumerate(zip(paths[b:b + 12], labels[b:b + 12])):
    shutil.copy(pth, f"results/demo_images/{i+1:02d}_{folders[lbl]}.jpg")
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")